In [266]:
import re
from pathlib import Path
from typing import Final

import ipywidgets as widgets
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

import tsam

# Purpose And Scope
This notebook starts from cleaned hourly input datasets, validates the data, prepares the calendar grouping required by the Option 1 approach, runs grouped TSAM aggregation, combines the reduced outputs, and inspects the resulting representative-period dataset.

## Method Overview
Option 1 applies the following procedure:

- split the year by calendar month
- classify each day as working or non-working
- cluster each month/day-type group separately
- select real observed days as representative periods
- assign each representative day a weight equal to the number of original days in its cluster
- concatenate the selected representative days into a reduced chronology

Default output structure under the baseline 5 working-day / 2 non-working-day cluster configuration:

- 24 clustering groups: 12 months x 2 day types
- 84 representative days: 12 x (5 working-day representatives + 2 non-working-day representatives)
- 2016 hourly snapshots: 84 x 24
- 12 monthly reduced blocks: each 7 x 24 = 168 hours

If the cluster-count policy is changed, the number of representative days and reduced hourly snapshots changes accordingly.

## Table of Contents

- [Purpose And Scope](#purpose-and-scope)
  - [Method Overview](#method-overview)
- [Data Path Configuration](#data-path-configuration)
- [Data Loading](#data-loading)
- [Snapshot Feature Handling](#snapshot-feature-handling)
- [Data Validation](#data-validation)
  - [Duplicates, NaNs, Numeric Columns & Full-Year Coverage](#duplicates-nans-numeric-columns--full-year-coverage)
  - [Capacity-Factor Range Validation](#capacity-factor-range-validation)
  - [Running Validation](#running-validation)
- [Building Daily Metadata](#building-daily-metadata)
  - [Metadata Creation](#metadata-creation)
  - [Metadata Validation](#metadata-validation)
- [TSAM Clustering](#tsam-clustering)
  - [Data Collection](#data-collection)
  - [Data Aggregation](#data-aggregation)
  - [Sanity Checks](#sanity-checks)
  - [Inspect Aggregation Outputs](#inspect-aggregation-outputs)
- [Diagnostic Visualizations](#diagnostic-visualizations)
  - [Calendar Heatmap Of Original Days By Assigned Representative](#calendar-heatmap-of-original-days-by-assigned-representative)
  - [Representative Weight Heatmap](#representative-weight-heatmap)
  - [Cluster Accuracy Overview](#cluster-accuracy-overview)
  - [Group-Level TSAM Diagnostic Drilldowns](#group-level-tsam-diagnostic-drilldowns)
- [Saving Results](#saving-results)

# Data Path Configuration

In [267]:
cur_location: Path = Path().resolve()
data_location: Path = cur_location / "../data"

# Data Loading


Specify the CSV separator through `sep`. The input datasets use both comma-separated and semicolon-separated formats.

In [268]:
demand = pd.read_csv(data_location / "Demand_ENTSO_E.csv", sep=";")
demand.head()

,snapshot,AL,AT,BA,BE,BG,CH,CZ,DE,DK,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,UK
0,01.01.2025 00:00,459,3871,609,6691,2374,4603,4029,28833,2606,...,9796,11525,11852,3591,3645,2578,9701,728,1823,18285
1,01.01.2025 01:00,428,3717,571,6327,2292,4711,3989,27781,2511,...,9535,11586,11284,3480,3532,2474,9551,699,1780,18154
2,01.01.2025 02:00,396,3522,539,6128,2237,4861,3885,26917,2426,...,9266,11540,10983,3289,3467,2316,9377,666,1733,16545
3,01.01.2025 03:00,376,3488,523,5982,2206,4810,3855,27016,2400,...,9013,11517,10827,3126,3438,2175,9417,648,1729,14918
4,01.01.2025 04:00,367,3598,526,5967,2194,4725,3865,26870,2405,...,9040,11602,10754,3007,3428,2059,9560,664,1724,14200


In [269]:
onwind_cf = pd.read_csv(data_location / "onwind_capacity_factors.csv", sep=",")
onwind_cf.head()

,snapshot,AL_onwind_2025,AT_onwind_2025,BA_onwind_2025,BE_onwind_2025,BG_onwind_2025,CH_onwind_2025,CZ_onwind_2025,DE_onwind_2025,DK_onwind_2025,...,NO_onwind_2025,PL_onwind_2025,PT_onwind_2025,RO_onwind_2025,RS_onwind_2025,SE_onwind_2025,SI_onwind_2025,SK_onwind_2025,UA_onwind_2025,XK_onwind_2025
0,01.01.2025 00:00,0.0,0.026443,0.0,0.968038,0.027777,0.030767,0.419450,0.716840,1.000000,...,0.131214,0.873780,0.116221,0.051165,0.0,0.297268,0.013709,0.114870,0.346401,0.0
1,01.01.2025 01:00,0.0,0.029090,0.0,0.975868,0.027310,0.030301,0.426459,0.728613,0.999992,...,0.109802,0.897004,0.109594,0.054343,0.0,0.286711,0.015315,0.114240,0.345689,0.0
2,01.01.2025 02:00,0.0,0.033465,0.0,0.983856,0.028493,0.029116,0.443365,0.743509,0.999972,...,0.092031,0.911984,0.105639,0.062091,0.0,0.278833,0.019344,0.110237,0.332158,0.0
3,01.01.2025 03:00,0.0,0.034560,0.0,0.988063,0.030633,0.029512,0.467903,0.760476,0.999996,...,0.078412,0.923292,0.107288,0.067657,0.0,0.271188,0.023585,0.110385,0.326958,0.0
4,01.01.2025 04:00,0.0,0.036894,0.0,0.987477,0.034158,0.031685,0.495288,0.771249,1.000000,...,0.071422,0.927705,0.103116,0.072680,0.0,0.259012,0.025262,0.112671,0.322901,0.0


In [270]:
hydro_inflow = pd.read_csv(
    data_location / "hydro_inflow_scaled_deduped_2025.csv", sep=","
)
hydro_inflow.head()

,snapshot,AT_hydro_2025,BA_hydro_2025,BE_hydro_2025,BG_hydro_2025,CH_hydro_2025,CZ_hydro_2025,DE_hydro_2025,ES_hydro_2025,FI_hydro_2025,...,PL_hydro_2025,PT_hydro_2025,RO_hydro_2025,RS_hydro_2025,SI_hydro_2025,UA_hydro_2025,UK_hydro_2025,XK_hydro_2025,SE_hydro_2025,AL_hydro_2025
0,01.01.2025 00:00,230,83,2,49,411,39,54,922,420,...,33,114,141,2,135,556,43,3,4570,155
1,01.01.2025 01:00,229,83,2,49,411,40,54,922,420,...,33,114,141,2,135,555,43,3,4569,152
2,01.01.2025 02:00,229,83,2,49,412,41,54,922,421,...,33,114,141,2,135,555,43,3,4562,151
3,01.01.2025 03:00,228,83,2,49,412,41,54,922,422,...,34,114,141,2,135,555,43,3,4555,151
4,01.01.2025 04:00,228,83,2,49,412,41,54,922,423,...,34,114,141,2,135,555,43,3,4545,150


In [271]:
ror_cf = pd.read_csv(data_location / "ror_capacity_factors.csv", sep=",")
ror_cf.head()

,snapshot,AL_ror_2025,AT_ror_2025,BA_ror_2025,BE_ror_2025,BG_ror_2025,CH_ror_2025,CZ_ror_2025,DE_ror_2025,EE_ror_2025,...,NO_ror_2025,PL_ror_2025,PT_ror_2025,RO_ror_2025,RS_ror_2025,SE_ror_2025,SI_ror_2025,SK_ror_2025,UK_ror_2025,XK_ror_2025
0,01.01.2025 00:00,0.06198,0.304260,0.113561,0.687213,0.132907,0.266242,0.128353,0.596077,0.0,...,0.592028,0.503166,0.140826,0.371941,0.428571,0.35421,0.090094,0.232892,0.19516,0.07694
1,01.01.2025 01:00,0.06074,0.300330,0.149166,0.687989,0.132323,0.264119,0.406737,0.591670,0.0,...,0.585834,0.503452,0.062204,0.369386,0.411529,0.35409,0.087915,0.231347,0.19512,0.07694
2,01.01.2025 02:00,0.06031,0.300783,0.155915,0.626787,0.129821,0.262051,0.430766,0.584802,0.0,...,0.578365,0.502625,0.059455,0.363477,0.338847,0.35360,0.087664,0.233002,0.19510,0.07682
3,01.01.2025 03:00,0.06016,0.306833,0.111967,0.609455,0.130377,0.260064,0.438508,0.577704,0.0,...,0.576665,0.502492,0.044515,0.359085,0.249123,0.35307,0.087212,0.230574,0.19506,0.07675
4,01.01.2025 04:00,0.05997,0.305347,0.111861,0.587503,0.135743,0.258158,0.438407,0.575100,0.0,...,0.580272,0.502750,0.096728,0.359085,0.246115,0.35229,0.087112,0.230022,0.19502,0.07684


In [272]:
solar_cf = pd.read_csv(data_location / "solar_capacity_factors.csv", sep=";")
solar_cf.head()

,snapshot,AL_solar_2025,AT_solar_2025,BA_solar_2025,BE_solar_2025,BG_solar_2025,CH_solar_2025,CZ_solar_2025,DE_solar_2025,DK_solar_2025,...,NO_solar_2025,PL_solar_2025,PT_solar_2025,RO_solar_2025,RS_solar_2025,SE_solar_2025,SI_solar_2025,SK_solar_2025,UA_solar_2025,XK_solar_2025
0,01.01.2025 00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,01.01.2025 01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,01.01.2025 02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,01.01.2025 03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,01.01.2025 04:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Rename demand columns to make them more interpretable:

In [273]:
demand = demand.rename(
    columns={c: f"{c}_demand_2025" for c in demand.columns if c != "snapshot"}
)
demand

,snapshot,AL_demand_2025,AT_demand_2025,BA_demand_2025,BE_demand_2025,BG_demand_2025,CH_demand_2025,CZ_demand_2025,DE_demand_2025,DK_demand_2025,...,NL_demand_2025,NO_demand_2025,PL_demand_2025,PT_demand_2025,RO_demand_2025,RS_demand_2025,SE_demand_2025,SI_demand_2025,SK_demand_2025,UK_demand_2025
0,01.01.2025 00:00,459,3871,609,6691,2374,4603,4029,28833,2606,...,9796,11525,11852,3591,3645,2578,9701,728,1823,18285
1,01.01.2025 01:00,428,3717,571,6327,2292,4711,3989,27781,2511,...,9535,11586,11284,3480,3532,2474,9551,699,1780,18154
2,01.01.2025 02:00,396,3522,539,6128,2237,4861,3885,26917,2426,...,9266,11540,10983,3289,3467,2316,9377,666,1733,16545
3,01.01.2025 03:00,376,3488,523,5982,2206,4810,3855,27016,2400,...,9013,11517,10827,3126,3438,2175,9417,648,1729,14918
4,01.01.2025 04:00,367,3598,526,5967,2194,4725,3865,26870,2405,...,9040,11602,10754,3007,3428,2059,9560,664,1724,14200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,31.12.2025 19:00,623,4743,913,7678,2916,5301,4831,32565,2788,...,10804,11969,14245,4818,4350,3474,10638,932,2096,27044
8756,31.12.2025 20:00,589,4518,871,7462,2826,5092,4782,31431,2639,...,10185,11741,13609,4395,4166,3266,10310,860,2032,25212
8757,31.12.2025 21:00,558,4563,817,7659,2792,4918,4688,31197,2533,...,9827,11408,13228,4050,4041,3164,10173,813,2017,23374
8758,31.12.2025 22:00,539,4362,759,7697,2701,4948,4497,30146,2438,...,9638,11166,12693,3893,3924,3157,9817,773,2028,20966


This workflow uses ALL countries in the given datasets.

In [274]:
# Name/data pairs used by the validation loop below.
data: list[tuple[str, pd.DataFrame]] = [
    ("demand", demand),
    ("solar_cf", solar_cf),
    ("onwind_cf", onwind_cf),
    ("ror_cf", ror_cf),
    ("hydro_inflow", hydro_inflow),
]

All country - specific data validation.

Make sure that there's no duplicate headers, no duplicate columns, and no pandas-mangled columns and minimal diagnostics showcasing which countries are absent for which features, etc.

In [275]:
RAW_SOURCES: Final[dict[str, tuple[Path, str]]] = {
    "demand": (data_location / "Demand_ENTSO_E.csv", ";"),
    "solar_cf": (data_location / "solar_capacity_factors.csv", ";"),
    "onwind_cf": (data_location / "onwind_capacity_factors.csv", ","),
    "ror_cf": (data_location / "ror_capacity_factors.csv", ","),
    "hydro_inflow": (data_location / "hydro_inflow_scaled_deduped_2025.csv", ";"),
}


def find_duplicate_headers(path: Path, sep: str) -> list[str]:
    """Return CSV header names that appear more than once."""
    header = path.read_text(encoding="utf-8").splitlines()[0].split(sep)
    return sorted({col for col in header if header.count(col) > 1})


def validate_loaded_columns(name: str, df: pd.DataFrame) -> None:
    """Validate columns before snapshot indexing can hide naming issues."""
    if "snapshot" not in df.columns:
        raise ValueError(f"{name}: missing snapshot column")

    duplicate_cols = df.columns[df.columns.duplicated()].tolist()
    if duplicate_cols:
        raise ValueError(f"{name}: duplicate loaded columns: {duplicate_cols}")

    mangled_cols = [col for col in df.columns if re.search(r"\.\d+$", col)]
    if mangled_cols:
        raise ValueError(f"{name}: pandas-mangled columns: {mangled_cols}")


def countries_from_columns(df: pd.DataFrame) -> set[str]:
    """Extract country prefixes from non-snapshot column names."""
    return {col.split("_")[0] for col in df.columns if col != "snapshot"}


for name, (path, sep) in RAW_SOURCES.items():
    duplicate_headers = find_duplicate_headers(path, sep)
    if duplicate_headers:
        raise ValueError(f"{name}: duplicate raw CSV headers: {duplicate_headers}")

for name, df in data:
    validate_loaded_columns(name, df)

country_sets = {name: countries_from_columns(df) for name, df in data}
all_countries = set().union(*country_sets.values())
dataset_coverage = pd.DataFrame(
    [
        {
            "dataset": name,
            "country_count": len(countries),
            "countries": ", ".join(sorted(countries)),
            "missing_from_union": ", ".join(sorted(all_countries - countries)) or "-",
        }
        for name, countries in country_sets.items()
    ]
)
dataset_coverage

,dataset,country_count,countries,missing_from_union
0,demand,34,"AL, AT, BA, BE, BG, CH, CZ, DE, DK, EE, ES, FI...","UA, XK"
1,solar_cf,36,"AL, AT, BA, BE, BG, CH, CZ, DE, DK, EE, ES, FI...",-
2,onwind_cf,36,"AL, AT, BA, BE, BG, CH, CZ, DE, DK, EE, ES, FI...",-
3,ror_cf,34,"AL, AT, BA, BE, BG, CH, CZ, DE, EE, ES, FI, FR...","DK, UA"
4,hydro_inflow,30,"AL, AT, BA, BE, BG, CH, CZ, DE, ES, FI, FR, GR...","DK, EE, LT, LV, NL, SK"


# Snapshot Feature Handling
TSAM requires snapshots to be stored as the DataFrame index, and that index must use a datetime dtype.

In [276]:
def set_new_index(
    df: pd.DataFrame, new_index_col: str = "snapshot"
) -> pd.DataFrame:
    """Set ``new_index_col`` as the DataFrame index in-place and return ``df``.

    The notebook keeps separate variables like ``demand`` and also stores the same
    DataFrame objects in ``data``. Mutating in-place keeps both references aligned.
    """
    if new_index_col not in df.columns:
        raise KeyError(
            f"{new_index_col!r} is not a column in the provided DataFrame"
        )

    df.set_index(new_index_col, drop=True, inplace=True)
    return df


# Set the snapshot column as the index for every source table.
data = [(df_name, set_new_index(df, "snapshot")) for df_name, df in data]

print("Index columns were swapped successfully!")

Index columns were swapped successfully!


Convert each snapshot index to datetime. The source timestamps use day-first dates; therefore, `dayfirst=True` is required.

In [277]:
# Converting index to datetime type
for df_name, df in data:
    df.index = pd.to_datetime(df.index, dayfirst=True)

print("Successfully converted indexes into datetime dtype!")

Successfully converted indexes into datetime dtype!


# Data Validation
This section validates the input data before aggregation. Failed checks raise explicit errors and identify the failing condition.

The validation step reduces the risk of silent TSAM failures or misleading representative periods.

Checks performed:

- no duplicate timestamps
- no missing values
- numeric columns only
- complete hourly year coverage
- capacity-factor columns remain within the expected `0.0 - 1.0` range

## Duplicates, NaNs, Numeric Columns & Full-Year Coverage
This validator checks the main TSAM input requirements: datetime index, complete hourly coverage, no duplicate timestamps, no missing values, and numeric columns only.

In [278]:
def validate_df(
    df: pd.DataFrame,
    name: str,
    year_to_check: int = 2025,
    period: str = "h",
) -> None:
    """Validate that a DataFrame is ready for hourly TSAM aggregation.

    TSAM expects a regular DatetimeIndex, numeric values, no gaps, no duplicates,
    and no NaNs. This function fails fast before aggregation can silently produce
    misleading representative periods.
    """
    expected_index = pd.date_range(
        start=f"{year_to_check}-01-01 00:00",
        end=f"{year_to_check}-12-31 23:00",
        freq=period,
    )

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: index must be a DatetimeIndex")

    if len(df) != len(expected_index):
        raise ValueError(
            f"{name}: expected {len(expected_index)} rows, got {len(df)}"
        )

    if df.index.has_duplicates:
        raise ValueError(f"{name}: index has duplicate timestamps")

    if not df.index.equals(expected_index):
        missing = expected_index.difference(df.index)
        extra = df.index.difference(expected_index)
        raise ValueError(
            f"{name}: index does not exactly match hourly {year_to_check}. "
            f"Missing examples: {missing[:5].tolist()}. "
            f"Extra examples: {extra[:5].tolist()}."
        )

    if df.isna().any().any():
        cols = df.columns[df.isna().any()].tolist()
        raise ValueError(f"{name}: contains NaNs in columns {cols}")

    non_numeric_cols = df.select_dtypes(exclude="number").columns.tolist()
    if non_numeric_cols:
        raise TypeError(
            f"{name}: non-numeric columns found: {non_numeric_cols}"
        )


## Capacity-Factor Range Validation
Capacity-factor features are expected to remain between `0.0` and `1.0`. This validator raises an error if any capacity-factor value falls outside that interval.

In [279]:
def validate_cf_range(df: pd.DataFrame, name: str) -> None:
    """Validate that capacity-factor columns stay inside the physical 0..1 range."""
    is_in_range = df.ge(0.0) & df.le(1.0)

    if not is_in_range.all().all():
        invalid_cols = df.columns[~is_in_range.all()].tolist()
        raise ValueError(
            f"{name}: values outside the 0.0-1.0 CF range in {invalid_cols}"
        )


## Running Validation
Run all validation checks. A failed check raises an error that identifies the violated condition.

In [280]:
capacity_factor_datasets: Final[set[str]] = {
    "solar_cf",
    "onwind_cf",
    "ror_cf",
}

for name, df in data:
    validate_df(df, name, year_to_check=2025, period="h")

    if name in capacity_factor_datasets:
        validate_cf_range(df, name)

print("Data validation has been successful!")

Data validation has been successful!


TSAM expects one dataset for the region being aggregated. The features are joined into a single DataFrame.

In [281]:
data: pd.DataFrame = pd.concat([df for _, df in data], axis=1)
duplicate_cols = data.columns[data.columns.duplicated()].tolist()
if duplicate_cols:
    raise ValueError(f"Joined data has duplicate columns: {duplicate_cols}")
data.head()

,AL_demand_2025,AT_demand_2025,BA_demand_2025,BE_demand_2025,BG_demand_2025,CH_demand_2025,CZ_demand_2025,DE_demand_2025,DK_demand_2025,EE_demand_2025,...,PL_hydro_2025,PT_hydro_2025,RO_hydro_2025,RS_hydro_2025,SI_hydro_2025,UA_hydro_2025,UK_hydro_2025,XK_hydro_2025,SE_hydro_2025,AL_hydro_2025
snapshot,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,459,3871,609,6691,2374,4603,4029,28833,2606,626,...,33,114,141,2,135,556,43,3,4570,155
2025-01-01 01:00:00,428,3717,571,6327,2292,4711,3989,27781,2511,598,...,33,114,141,2,135,555,43,3,4569,152
2025-01-01 02:00:00,396,3522,539,6128,2237,4861,3885,26917,2426,583,...,33,114,141,2,135,555,43,3,4562,151
2025-01-01 03:00:00,376,3488,523,5982,2206,4810,3855,27016,2400,578,...,34,114,141,2,135,555,43,3,4555,151
2025-01-01 04:00:00,367,3598,526,5967,2194,4725,3865,26870,2405,580,...,34,114,141,2,135,555,43,3,4545,150


# Building Daily Metadata
The Option 1 method clusters days separately by calendar month and by working/non-working status. Calendar metadata is therefore added to the hourly dataset before aggregation.

This section creates two table grains:

- `hourly_data_with_metadata`: hourly table with calendar metadata, used later for TSAM group slicing
- `daily_metadata`: one-row-per-day table, used only for calendar validation

## Metadata Creation
Create the calendar fields required to form the 24 clustering groups:

- date
- month
- day of month
- weekday
- working/non-working label
- group ID, such as `2025_1_working` or `2025_1_non-working`

Assumptions:

- Saturday and Sunday are classified as non-working days.
- Public holidays are excluded unless a holiday calendar is supplied.

In [282]:
# Baseline: weekends only, no holidays.
NON_WORKING_WEEKDAYS: Final[set[str]] = {"Saturday", "Sunday"}

# Hourly table + calendar metadata.
hourly_data_with_metadata = data.copy()
snapshot_index = hourly_data_with_metadata.index

hourly_data_with_metadata["date"] = snapshot_index.date
hourly_data_with_metadata["month"] = snapshot_index.month
hourly_data_with_metadata["day_of_month"] = snapshot_index.day
hourly_data_with_metadata["weekday"] = snapshot_index.day_name()

# Default to working days, then override weekends.
hourly_data_with_metadata["day_type"] = "working"
hourly_data_with_metadata.loc[
    hourly_data_with_metadata["weekday"].isin(NON_WORKING_WEEKDAYS),
    "day_type",
] = "non-working"

# Key for the 24 TSAM groups, e.g. 2025_1_working.
hourly_data_with_metadata["group_id"] = (
    pd.Series(snapshot_index.year, index=snapshot_index).astype(str)
    + "_"
    + hourly_data_with_metadata["month"].astype(str)
    + "_"
    + hourly_data_with_metadata["day_type"]
)
hourly_data_with_metadata.head()

,AL_demand_2025,AT_demand_2025,BA_demand_2025,BE_demand_2025,BG_demand_2025,CH_demand_2025,CZ_demand_2025,DE_demand_2025,DK_demand_2025,EE_demand_2025,...,UK_hydro_2025,XK_hydro_2025,SE_hydro_2025,AL_hydro_2025,date,month,day_of_month,weekday,day_type,group_id
snapshot,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,459,3871,609,6691,2374,4603,4029,28833,2606,626,...,43,3,4570,155,2025-01-01,1,1,Wednesday,working,2025_1_working
2025-01-01 01:00:00,428,3717,571,6327,2292,4711,3989,27781,2511,598,...,43,3,4569,152,2025-01-01,1,1,Wednesday,working,2025_1_working
2025-01-01 02:00:00,396,3522,539,6128,2237,4861,3885,26917,2426,583,...,43,3,4562,151,2025-01-01,1,1,Wednesday,working,2025_1_working
2025-01-01 03:00:00,376,3488,523,5982,2206,4810,3855,27016,2400,578,...,43,3,4555,151,2025-01-01,1,1,Wednesday,working,2025_1_working
2025-01-01 04:00:00,367,3598,526,5967,2194,4725,3865,26870,2405,580,...,43,3,4545,150,2025-01-01,1,1,Wednesday,working,2025_1_working


## Metadata Validation
Validate the calendar split before running TSAM. An invalid split can still produce clusters, but the clusters would represent incorrect day classes.

This section checks three conditions:

- weekday/weekend classification follows the baseline rule
- the day-level metadata covers the full 365-day year
- all 24 month/day-type groups are feasible for the requested cluster counts

In [283]:
# One row per original day for calendar validation.
daily_metadata = (
    hourly_data_with_metadata.assign(
        date=hourly_data_with_metadata.index.normalize()
    )
    .drop_duplicates(subset="date")
    .set_index("date")
)

weekday_order: Final[list[str]] = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
working_weekdays: Final[list[str]] = [
    weekday for weekday in weekday_order if weekday not in NON_WORKING_WEEKDAYS
]
day_type_order: Final[list[str]] = ["working", "non-working"]

# Validate weekday/weekend classification.
weekday_day_type_counts = pd.crosstab(
    daily_metadata["weekday"], daily_metadata["day_type"]
).reindex(index=weekday_order, columns=day_type_order, fill_value=0)
weekday_day_type_counts

day_type,working,non-working
weekday,,
Monday,52,0
Tuesday,52,0
Wednesday,53,0
Thursday,52,0
Friday,52,0
Saturday,0,52
Sunday,0,52


The crosstab provides an interpretable classification check: Monday-Friday should appear only under `working`, and Saturday/Sunday should appear only under `non-working`. The assertions encode the same rule and confirm that no day was lost while collapsing hourly data to daily metadata.

In [284]:
# Weekday/weekend classification rule.
assert (weekday_day_type_counts.loc[working_weekdays, "non-working"] == 0).all()
assert (
    weekday_day_type_counts.loc[list(NON_WORKING_WEEKDAYS), "working"] == 0
).all()

# Full non-leap year coverage.
expected_days_in_year: Final[int] = 365
total_days = len(daily_metadata)
assert total_days == expected_days_in_year, (
    f"The day-level metadata should contain {expected_days_in_year} days, got: {total_days}"
)

# Crosstab should account for the same day rows.
assert weekday_day_type_counts.to_numpy().sum() == total_days
print("The validation has been passed!")

The validation has been passed!


The final table checks the 24 groups used by the aggregation loop. Each month must contain enough working and non-working days to select the requested number of medoids.

In [285]:
# Requested representative days per month/day-type group.
requested_clusters_by_day_type: Final[dict[str, int]] = {
    "working": 5,
    "non-working": 2,
}
month_order: Final[range] = range(1, 13)

# Count real days in each of the 24 groups.
month_day_type_counts = (
    daily_metadata.groupby(["month", "day_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=month_order,
        columns=requested_clusters_by_day_type.keys(),
        fill_value=0,
    )
)

# 12 months x 2 day types.
expected_group_count = len(month_order) * len(requested_clusters_by_day_type)
assert daily_metadata["group_id"].nunique() == expected_group_count

# Each group must have enough days for its requested clusters.
has_enough_days_for_clusters = month_day_type_counts.ge(
    pd.Series(requested_clusters_by_day_type),
    axis="columns",
)
assert has_enough_days_for_clusters.all().all(), (
    "Each month/day-type group must have at least as many days as requested clusters"
)

month_day_type_counts

day_type,working,non-working
month,,
1,23,8
2,20,8
3,21,10
4,22,8
5,22,9
6,21,9
7,23,8
8,21,10
9,22,8


# TSAM Clustering
Option 1 is implemented as 24 separate TSAM aggregations: one for each month/day-type group.

For each group, the workflow:

- passes only that group's hourly data into TSAM
- uses 24-hour daily periods
- requests the configured number of medoids for that group
- uses medoid representation so selected days are real observed days
- excludes extreme-day enrichment in this baseline implementation

For each group, the workflow stores:

- reduced hourly rows for the selected medoid days
- original day to assigned cluster mapping
- cluster weights
- the raw TSAM result object for diagnostics

The combined reduced-hourly row count is determined by the number of selected representative days multiplied by 24.

[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/api/#tsam.api.aggregate)

## Data Collection
Start with the 24 group IDs created from the metadata section. The order is chronological by year and month, with working groups before non-working groups.

In [286]:
DAY_TYPE_SORT_ORDER: Final[dict[str, int]] = {"working": 0, "non-working": 1}


def sort_group_id(group_id: str) -> tuple[int, int, int]:
    """Sort groups by year, month, then working/non-working day type."""
    year, month, day_type = group_id.split("_", maxsplit=2)
    return int(year), int(month), DAY_TYPE_SORT_ORDER[day_type]


group_ids = sorted(
    hourly_data_with_metadata["group_id"].unique(),
    key=sort_group_id,
)
group_ids

['2025_1_working',
 '2025_1_non-working',
 '2025_2_working',
 '2025_2_non-working',
 '2025_3_working',
 '2025_3_non-working',
 '2025_4_working',
 '2025_4_non-working',
 '2025_5_working',
 '2025_5_non-working',
 '2025_6_working',
 '2025_6_non-working',
 '2025_7_working',
 '2025_7_non-working',
 '2025_8_working',
 '2025_8_non-working',
 '2025_9_working',
 '2025_9_non-working',
 '2025_10_working',
 '2025_10_non-working',
 '2025_11_working',
 '2025_11_non-working',
 '2025_12_working',
 '2025_12_non-working']

These helper functions prepare one group at a time.

They maintain two versions of the same group:

- feature-only data passed into TSAM
- feature and metadata data used to map TSAM outputs back to original days

In [287]:
# Original feature columns.
FEATURE_COLUMNS = data.columns
WORKING_CLUSTERS_PER_MONTH = 5
NON_WORKING_CLUSTERS_PER_MONTH = 2


def add_group_day_number(
    group_data_with_metadata: pd.DataFrame,
) -> pd.DataFrame:
    """Add a 1-based day number inside one month/day-type group."""
    group_data_with_metadata["group_day_number"] = (
        group_data_with_metadata["day_of_month"]
        .ne(group_data_with_metadata["day_of_month"].shift())
        .cumsum()
    )
    return group_data_with_metadata


def slice_group_data(
    group_id: str,
    data_with_metadata: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Slice one month/day-type group and return TSAM features plus metadata."""
    # Keep metadata for later mapping.
    group_data_with_metadata = data_with_metadata.loc[
        data_with_metadata["group_id"] == group_id
    ].copy()
    group_data_with_metadata = add_group_day_number(group_data_with_metadata)

    # TSAM input: numeric feature columns only.
    group_features = group_data_with_metadata.loc[:, FEATURE_COLUMNS]
    return group_features, group_data_with_metadata


def get_n_clusters_for_group(group_data_with_metadata: pd.DataFrame) -> int:
    """Return requested cluster count for one month/day-type group."""
    day_types = group_data_with_metadata["day_type"].unique()

    if len(day_types) != 1:
        raise ValueError(f"Expected one day_type per group, got: {day_types}")

    day_type = day_types[0]

    if day_type == "working":
        return WORKING_CLUSTERS_PER_MONTH
    elif day_type == "non-working":
        return NON_WORKING_CLUSTERS_PER_MONTH

    raise ValueError(f"Unknown day_type: {day_type}")


def collect_representative_day_data(
    aggregation_result: tsam.AggregationResult,
    group_data_with_metadata: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build reduced hourly data and original-day assignments for one group."""
    representative_day_chunks = []
    # TSAM cluster centers are 0-based day positions inside this group.
    representative_day_indices = aggregation_result.clustering.cluster_centers
    cluster_weights_by_id = aggregation_result.cluster_weights

    # One row per original day: which cluster it belongs to.
    day_assignments = (
        aggregation_result.assignments.assign(
            date=aggregation_result.assignments.index.normalize()
        )
        .drop_duplicates("date")
        .set_index("date")
    )
    day_assignments = day_assignments.loc[:, ["cluster_idx"]].rename(
        columns={"cluster_idx": "cluster_id"}
    )
    day_assignments["cluster_weight"] = day_assignments["cluster_id"].map(
        lambda cluster_id: cluster_weights_by_id[cluster_id]
    )
    # Same cluster IDs repeat across groups, so keep the group key.
    day_assignments["group_id"] = group_data_with_metadata["group_id"]
    # Global representative ID: group key + local cluster ID.
    day_assignments["representative_id"] = (
        day_assignments["group_id"].astype(str)
        + "_c"
        + day_assignments["cluster_id"].astype(str)
    )

    cluster_ids = sorted(cluster_weights_by_id)
    for cluster_id, representative_day_index in zip(
        cluster_ids,
        representative_day_indices,
    ):
        # Slice the selected medoid day from the original group data.
        representative_day_hours = group_data_with_metadata.loc[
            group_data_with_metadata["group_day_number"]
            == representative_day_index + 1
        ].copy()

        representative_day_hours["cluster_id"] = cluster_id
        representative_day_hours["cluster_weight"] = cluster_weights_by_id[
            cluster_id
        ]

        current_group_id = representative_day_hours["group_id"].iloc[0]
        representative_day_hours["representative_id"] = (
            f"{current_group_id}_c{cluster_id}"
        )

        representative_day_chunks.append(representative_day_hours)

    # Concatenate all representative days from this group.
    group_reduced_hourly_data = pd.concat(representative_day_chunks)
    return group_reduced_hourly_data, day_assignments

## Data Aggregation
Run TSAM once for each group and combine the per-group outputs.

Important settings:

- `period_duration=24`: each TSAM period is one day
- `representation="medoid"`: representatives are real observed days
- `preserve_column_means=False`: medoid profiles are not rescaled away from the original observed values

The aggregation returns three objects:

- `reduced_hourly_df`: selected medoid days, one row per reduced hour
- `day_assignments_df`: one row per original day, showing its assigned cluster
- `tsam_results_by_group`: raw TSAM results keyed by group ID

In [288]:
BASELINE_CLUSTER_CONFIG = tsam.ClusterConfig(
    method="hierarchical",
    representation="medoid",
)
PERIOD_DURATION_HOURS: Final[int] = 24
TSAM_NUMERICAL_TOLERANCE: Final[float] = 1e-9


def run_aggregation_all_groups(
    group_ids: list[str],
    data_with_metadata: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, tsam.AggregationResult]]:
    """Run one TSAM aggregation per group and combine the outputs."""
    reduced_hourly_chunks: list[pd.DataFrame] = []
    day_assignment_chunks: list[pd.DataFrame] = []
    tsam_results_by_group: dict[str, tsam.AggregationResult] = {}

    for group_id in group_ids:
        # Slice this group into TSAM features and metadata-rich source data.
        group_features, group_data_with_metadata = slice_group_data(
            group_id,
            data_with_metadata,
        )

        # Working groups get 5 medoids; non-working groups get 2.
        n_clusters = get_n_clusters_for_group(group_data_with_metadata)

        aggregation_result = tsam.aggregate(
            data=group_features,
            n_clusters=n_clusters,
            period_duration=PERIOD_DURATION_HOURS,
            preserve_column_means=False,
            cluster=BASELINE_CLUSTER_CONFIG,
            numerical_tolerance=TSAM_NUMERICAL_TOLERANCE,
        )

        tsam_results_by_group[group_id] = aggregation_result

        group_reduced_hourly_data, group_day_assignments = (
            collect_representative_day_data(
                aggregation_result,
                group_data_with_metadata,
            )
        )
        reduced_hourly_chunks.append(group_reduced_hourly_data)
        day_assignment_chunks.append(group_day_assignments)

    reduced_hourly_data = pd.concat(reduced_hourly_chunks).sort_index()
    day_assignments_data = pd.concat(day_assignment_chunks).sort_index()

    return reduced_hourly_data, day_assignments_data, tsam_results_by_group

Apply the aggregation. This table contains only selected medoid days, not a reconstructed 365-day year.

In [289]:
reduced_hourly_df, day_assignments_df, tsam_results_by_group = (
    run_aggregation_all_groups(group_ids, hourly_data_with_metadata)
)

Build a compact representative-day summary table. The table has one row per selected medoid day, so its shape follows the cluster-count policy used for the grouped TSAM runs.

In [290]:
representative_day_columns: Final[list[str]] = [
    "selected_medoid_date",
    "representative_id",
    "group_id",
    "month",
    "day_type",
    "cluster_id",
    "cluster_weight",
]

representative_days = (
    reduced_hourly_df.drop_duplicates(subset="representative_id", keep="first")
    .rename(columns={"date": "selected_medoid_date"})
    .loc[:, representative_day_columns]
    .sort_values("selected_medoid_date")
    .reset_index(drop=True)
)

## Sanity Checks
These checks validate the structure of the reduced dataset before visualization or downstream modelling.

They verify five conditions:

- expected fixed object sizes: 365 assigned original days and 24 TSAM groups
- cluster-policy compliance: each group has the configured number of representatives
- representative structure: each selected representative has exactly 24 hourly rows
- assignment integrity: every original day is assigned exactly once
- weight consistency: representative weights match the number of assigned original days

In [291]:
expected_original_days: Final[int] = 365
expected_group_count: Final[int] = 12 * 2

# Expected fixed object sizes.
assert len(day_assignments_df) == expected_original_days
assert len(tsam_results_by_group) == expected_group_count

# Source of truth: the configured cluster policy applied to each original group.
expected_cluster_counts_by_group = pd.Series(
    {
        group_id: get_n_clusters_for_group(
            hourly_data_with_metadata.loc[
                hourly_data_with_metadata["group_id"] == group_id
            ]
        )
        for group_id in group_ids
    },
    name="expected_n_clusters",
)

# The reduced outputs must match the configured policy, not merely TSAM internals.
representative_counts_by_group = representative_days.groupby("group_id").size()
assert representative_counts_by_group.sort_index().equals(
    expected_cluster_counts_by_group.sort_index()
)

# Secondary consistency check: the stored TSAM result objects should report the same policy.
result_cluster_counts = pd.Series(
    {
        group_id: aggregation_result.n_clusters
        for group_id, aggregation_result in tsam_results_by_group.items()
    },
    name="n_clusters",
)
assert result_cluster_counts.sort_index().equals(
    expected_cluster_counts_by_group.sort_index()
)

expected_representative_days = int(expected_cluster_counts_by_group.sum())
expected_reduced_hours = expected_representative_days * PERIOD_DURATION_HOURS

assert len(representative_days) == expected_representative_days
assert (
    reduced_hourly_df["representative_id"].nunique()
    == expected_representative_days
)
assert len(reduced_hourly_df) == expected_reduced_hours
assert (
    reduced_hourly_df["representative_id"].value_counts()
    == PERIOD_DURATION_HOURS
).all()

# Assignment integrity.
assert day_assignments_df.index.is_unique
assert day_assignments_df["cluster_id"].notna().all()
assert set(day_assignments_df["representative_id"]) == set(
    reduced_hourly_df["representative_id"]
)

# Weight consistency.
assert representative_days["cluster_weight"].sum() == expected_original_days

assigned_day_counts = day_assignments_df["representative_id"].value_counts()
representative_weights = representative_days.set_index("representative_id")[
    "cluster_weight"
]
assert assigned_day_counts.sort_index().equals(
    representative_weights.sort_index()
)

print("The validation has been passed!")

The validation has been passed!


The reduced chronology has the expected shape, and every original day maps to exactly one representative day.

## Inspect Aggregation Outputs
Before visualization, inspect the three normalized output tables and one raw TSAM result.

The normalized tables are the main interface for the rest of the notebook:

- `representative_days`: one row per selected medoid day
- `day_assignments_df`: one row per original day
- `reduced_hourly_df`: one row per selected medoid hour

The raw TSAM result is useful for diagnostics for one group, but it is not the primary plotting interface.

[TSAM result API reference](https://tsam.readthedocs.io/en/latest/api/tsam/result/#tsam.result.AggregationResult)

### Output Inventory
This inventory states the table grain and purpose before row-level inspection.

In [292]:
def render_wrapped_table(df: pd.DataFrame, text_columns: list[str]):
    """Render explanation tables with wrapped long-text columns."""
    text_column_style = {
        "white-space": "normal",
        "text-align": "left",
        "min-width": "280px",
        "max-width": "520px",
    }

    return df.style.set_properties(**{"text-align": "left"}).set_properties(
        subset=text_columns, **text_column_style
    )


output_inventory = pd.DataFrame(
    [
        {
            "object": "representative_days",
            "grain": "one selected medoid day",
            "rows": len(representative_days),
            "columns": representative_days.shape[1],
            "main_use": "medoid date table and cluster weight summaries",
        },
        {
            "object": "day_assignments_df",
            "grain": "one original day",
            "rows": len(day_assignments_df),
            "columns": day_assignments_df.shape[1],
            "main_use": "calendar plots and cluster member tables",
        },
        {
            "object": "reduced_hourly_df",
            "grain": "one selected medoid hour",
            "rows": len(reduced_hourly_df),
            "columns": reduced_hourly_df.shape[1],
            "main_use": "reduced hourly time series for downstream modelling",
        },
        {
            "object": "tsam_results_by_group",
            "grain": "one raw TSAM result per group",
            "rows": len(tsam_results_by_group),
            "columns": "-",
            "main_use": "diagnostics and TSAM internals",
        },
    ]
)
render_wrapped_table(output_inventory, text_columns=["main_use"])

,object,grain,rows,columns,main_use
0,representative_days,one selected medoid day,84,7,medoid date table and cluster weight summaries
1,day_assignments_df,one original day,365,4,calendar plots and cluster member tables
2,reduced_hourly_df,one selected medoid hour,2016,180,reduced hourly time series for downstream modelling
3,tsam_results_by_group,one raw TSAM result per group,24,-,diagnostics and TSAM internals


### Representative Days
This is the main day-level summary table. Each row is one selected medoid day, and `cluster_weight` indicates how many original days it represents.

In [293]:
representative_days

,selected_medoid_date,representative_id,group_id,month,day_type,cluster_id,cluster_weight
0,2025-01-04,2025_1_non-working_c0,2025_1_non-working,1,non-working,0,6
1,2025-01-07,2025_1_working_c0,2025_1_working,1,working,0,8
2,2025-01-16,2025_1_working_c2,2025_1_working,1,working,2,5
3,2025-01-22,2025_1_working_c1,2025_1_working,1,working,1,5
4,2025-01-25,2025_1_non-working_c1,2025_1_non-working,1,non-working,1,2
...,...,...,...,...,...,...,...
79,2025-12-14,2025_12_non-working_c0,2025_12_non-working,12,non-working,0,6
80,2025-12-19,2025_12_working_c4,2025_12_working,12,working,4,5
81,2025-12-23,2025_12_working_c0,2025_12_working,12,working,0,5
82,2025-12-27,2025_12_non-working_c1,2025_12_non-working,12,non-working,1,2


### Day Assignments
This table maps each original date to its assigned representative. It is the base table for calendar assignment plots and cluster member summaries.

In [294]:
day_assignments_df

,cluster_id,cluster_weight,group_id,representative_id
date,,,,
2025-01-01,0,8,2025_1_working,2025_1_working_c0
2025-01-02,0,8,2025_1_working,2025_1_working_c0
2025-01-03,0,8,2025_1_working,2025_1_working_c0
2025-01-04,0,6,2025_1_non-working,2025_1_non-working_c0
2025-01-05,0,6,2025_1_non-working,2025_1_non-working_c0
...,...,...,...,...
2025-12-27,1,2,2025_12_non-working,2025_12_non-working_c1
2025-12-28,1,2,2025_12_non-working,2025_12_non-working_c1
2025-12-29,2,3,2025_12_working,2025_12_working_c2


### Reduced Hourly Data
This is the reduced hourly table containing selected medoid hours only. The preview includes identifiers, weights, and the numeric time-series columns.

In [295]:
reduced_hourly_preview = reduced_hourly_df.reset_index()
reduced_hourly_preview_columns: Final[list[str]] = [
    "snapshot",
    "representative_id",
    "cluster_weight",
    *FEATURE_COLUMNS,
]

reduced_hourly_preview.loc[:, reduced_hourly_preview_columns].head(10)

,snapshot,representative_id,cluster_weight,AL_demand_2025,AT_demand_2025,BA_demand_2025,BE_demand_2025,BG_demand_2025,CH_demand_2025,CZ_demand_2025,...,PL_hydro_2025,PT_hydro_2025,RO_hydro_2025,RS_hydro_2025,SI_hydro_2025,UA_hydro_2025,UK_hydro_2025,XK_hydro_2025,SE_hydro_2025,AL_hydro_2025
0,2025-01-04 00:00:00,2025_1_non-working_c0,6,488,4405,624,7805,2619,4858,5296,...,40,115,142,4,135,564,43,3,4493,186
1,2025-01-04 01:00:00,2025_1_non-working_c0,6,428,4312,596,7528,2602,4977,5226,...,40,115,142,4,135,564,43,3,4499,185
2,2025-01-04 02:00:00,2025_1_non-working_c0,6,407,4211,586,7373,2634,5275,5157,...,40,115,142,4,135,564,43,3,4505,185
3,2025-01-04 03:00:00,2025_1_non-working_c0,6,434,4299,595,7403,2729,5397,5264,...,40,115,142,4,135,563,43,3,4512,184
4,2025-01-04 04:00:00,2025_1_non-working_c0,6,420,4619,635,7635,2971,5079,5616,...,40,115,142,4,135,563,43,3,4518,184
5,2025-01-04 05:00:00,2025_1_non-working_c0,6,474,5120,739,8212,3258,5000,6308,...,40,115,142,4,135,563,43,3,4527,183
6,2025-01-04 06:00:00,2025_1_non-working_c0,6,489,5688,821,8713,3513,4851,6637,...,40,115,142,4,135,563,43,3,4538,183
7,2025-01-04 07:00:00,2025_1_non-working_c0,6,547,5944,880,9225,3632,4983,6734,...,40,115,142,4,135,563,43,3,4550,182
8,2025-01-04 08:00:00,2025_1_non-working_c0,6,581,6127,913,9365,3614,5190,6904,...,40,115,142,4,135,562,43,3,4562,182
9,2025-01-04 09:00:00,2025_1_non-working_c0,6,600,6187,912,9478,3577,5306,6910,...,40,115,142,4,135,562,43,3,4571,181


### Example TSAM Result
The workflow ran 24 independent TSAM aggregations. This section inspects one chronological group to illustrate the raw result object without printing all 24 groups.

In [296]:
example_group_id = representative_days.loc[0, "group_id"]
example_features, example_group_data_with_metadata = slice_group_data(
    example_group_id, hourly_data_with_metadata
)
example_result = tsam_results_by_group[example_group_id]

print(f"Example group: {example_group_id}")

Example group: 2025_1_non-working


#### Raw Object Inventory
These are the raw objects TSAM exposes for the selected example group.

In [297]:
example_result_inventory = pd.DataFrame(
    [
        {
            "object": "cluster_representatives",
            "shape": str(example_result.cluster_representatives.shape),
            "meaning": "representative profile per cluster and hour",
        },
        {
            "object": "cluster_assignments",
            "shape": str(example_result.cluster_assignments.shape),
            "meaning": "cluster id for each original day in the group",
        },
        {
            "object": "assignments",
            "shape": str(example_result.assignments.shape),
            "meaning": "same assignments with period/hour index columns",
        },
        {
            "object": "cluster_weights",
            "shape": str(pd.Series(example_result.cluster_weights).shape),
            "meaning": "number of original days represented by each cluster",
        },
        {
            "object": "reconstructed",
            "shape": str(example_result.reconstructed.shape),
            "meaning": "group-length reconstruction using representative profiles",
        },
        {
            "object": "residuals",
            "shape": str(example_result.residuals.shape),
            "meaning": "pointwise reconstruction errors",
        },
    ]
)
render_wrapped_table(example_result_inventory, text_columns=["meaning"])

,object,shape,meaning
0,cluster_representatives,"(48, 170)",representative profile per cluster and hour
1,cluster_assignments,"(8,)",cluster id for each original day in the group
2,assignments,"(192, 3)",same assignments with period/hour index columns
3,cluster_weights,"(2,)",number of original days represented by each cluster
4,reconstructed,"(192, 170)",group-length reconstruction using representative profiles
5,residuals,"(192, 170)",pointwise reconstruction errors


#### Example Cluster Representatives
The index grain is `(cluster_id, hour_of_day)`. With `representation="medoid"`, each representative profile is copied from a real original day in the group.

In [298]:
example_result.cluster_representatives.head(10)

AL_demand_2025  AL_hydro_2025  AL_onwind_2025  AL_ror_2025  \
  timestep                                                               
0 0                  488.0          186.0        0.113264      0.07413   
  1                  428.0          185.0        0.083006      0.07393   
  2                  407.0          185.0        0.062000      0.07374   
  3                  434.0          184.0        0.041521      0.07355   
  4                  420.0          184.0        0.030790      0.07337   
  5                  474.0          183.0        0.025235      0.07319   
  6                  489.0          183.0        0.025106      0.07302   
  7                  547.0          182.0        0.025876      0.07285   
  8                  581.0          182.0        0.022825      0.07268   
  9                  600.0          181.0        0.020214      0.07252   

            AL_solar_2025  AT_demand_2025  AT_hydro_2025  AT_onwind_2025  \
  timestep                                                                 
0 0              0.000000          4405.0          219.0        0.201662   
  1              0.000000          4312.0          219.0        0.192658   
  2              0.000000          4211.0          219.0        0.187942   
  3              0.000000          4299.0          219.0        0.170759   
  4              0.000000          4619.0          219.0        0.158332   
  5              0.000000          5120.0          219.0        0.137685   
  6              0.000000          5688.0          219.0        0.100453   
  7              0.023770          5944.0          219.0        0.070848   
  8              0.077892          6127.0          219.0        0.047358   
  9              0.173073          6187.0          219.0        0.032979   

            AT_ror_2025  AT_solar_2025  ...  UA_solar_2025  UK_demand_2025  \
  timestep                              ...                                  
0 0            0.285512       0.000000  ...       0.000000         21002.0   
  1            0.272398       0.000000  ...       0.000000         20281.0   
  2            0.269771       0.000000  ...       0.000000         19492.0   
  3            0.273883       0.000000  ...       0.000000         19077.0   
  4            0.276763       0.000000  ...       0.000000         18739.0   
  5            0.282234       0.000000  ...       0.000000         19746.0   
  6            0.305510       0.000000  ...       0.000000         23173.0   
  7            0.330363       0.000000  ...       0.060170         26217.0   
  8            0.332554       0.111660  ...       0.175276         28862.0   
  9            0.299261       0.266636  ...       0.280054         30664.0   

            UK_hydro_2025  UK_onwind_2025  UK_ror_2025  UK_solar_2025  \
  timestep                                                              
0 0                  43.0        0.096394      0.19337            0.0   
  1                  43.0        0.110144      0.19335            0.0   
  2                  43.0        0.123004      0.19332            0.0   
  3                  43.0        0.130422      0.19329            0.0   
  4                  43.0        0.131729      0.19327            0.0   
  5                  43.0        0.131757      0.19324            0.0   
  6                  43.0        0.130627      0.19322            0.0   
  7                  43.0        0.131880      0.19319            0.0   
  8                  43.0        0.140885      0.19316            0.0   
  9                  43.0        0.149895      0.19314            0.0   

            XK_hydro_2025  XK_onwind_2025  XK_ror_2025  XK_solar_2025  
  timestep                                                             
0 0                   3.0        0.050828      0.07856       0.000000  
  1                   3.0        0.037231      0.07855       0.000000  
  2                   3.0        0.022550      0.07854       0.000000  
  3                   3.0        0.0

#### Day Assignments For The Example Group
The normalized assignment table is preferable for analysis because it contains real dates and global representative IDs.

In [299]:
example_group_day_assignments = day_assignments_df.loc[
    day_assignments_df["group_id"] == example_group_id
]
example_group_day_assignments

,cluster_id,cluster_weight,group_id,representative_id
date,,,,
2025-01-04,0,6,2025_1_non-working,2025_1_non-working_c0
2025-01-05,0,6,2025_1_non-working,2025_1_non-working_c0
2025-01-11,0,6,2025_1_non-working,2025_1_non-working_c0
2025-01-12,0,6,2025_1_non-working,2025_1_non-working_c0
2025-01-18,0,6,2025_1_non-working,2025_1_non-working_c0
2025-01-19,0,6,2025_1_non-working,2025_1_non-working_c0
2025-01-25,1,2,2025_1_non-working,2025_1_non-working_c1
2025-01-26,1,2,2025_1_non-working,2025_1_non-working_c1


TSAM's raw `assignments` table is lower-level. `period_idx` is the day number inside the group, and `timestep_idx` is the hour inside that day.

In [300]:
example_result.assignments.head(24)

,period_idx,timestep_idx,cluster_idx
snapshot,,,
2025-01-04 00:00:00,0,0,0
2025-01-04 01:00:00,0,1,0
2025-01-04 02:00:00,0,2,0
2025-01-04 03:00:00,0,3,0
2025-01-04 04:00:00,0,4,0
2025-01-04 05:00:00,0,5,0
2025-01-04 06:00:00,0,6,0
2025-01-04 07:00:00,0,7,0
2025-01-04 08:00:00,0,8,0


#### Example Cluster Weights
Cluster weights show how many original days each representative day represents.

In [301]:
example_cluster_weights = (
    pd.Series(example_result.cluster_weights, name="cluster_weight")
    .rename_axis("cluster_id")
    .reset_index()
)
example_cluster_weights

,cluster_id,cluster_weight
0,0,6
1,1,2


### Example Accuracy Tables
These TSAM metrics compare the original example group against TSAM's reconstructed version of the same group. They are diagnostic outputs, not the final reduced input.

In [302]:
example_accuracy_by_feature = pd.concat(
    [
        example_result.accuracy.rmse.rename("rmse"),
        example_result.accuracy.mae.rename("mae"),
        example_result.accuracy.rmse_duration.rename("rmse_duration"),
    ],
    axis=1,
)
example_accuracy_by_feature

,rmse,mae,rmse_duration
AL_demand_2025,0.094957,0.065359,0.060837
AL_hydro_2025,0.118971,0.084968,0.116375
AL_onwind_2025,0.252818,0.117525,0.191186
AL_ror_2025,0.118975,0.084974,0.116414
AL_solar_2025,0.161024,0.071551,0.051455
...,...,...,...
UK_solar_2025,0.116083,0.036883,0.061616
XK_hydro_2025,0.088510,0.044508,0.087532
XK_onwind_2025,0.330659,0.184280,0.196611
XK_ror_2025,0.087132,0.048582,0.084561


#### Reconstructed Data And Residuals
The reconstructed table has the original group length, where each original day has been replaced by its representative profile. Residuals are diagnostic outputs only; Option 1's reduced output remains `reduced_hourly_df`.

In [303]:
example_reconstruction_preview = pd.concat(
    {
        "reconstructed": example_result.reconstructed.head(5),
        "residuals": example_result.residuals.head(5),
    },
    axis=1,
)
example_reconstruction_preview

reconstructed                                           \
                    AL_demand_2025 AL_hydro_2025 AL_onwind_2025 AL_ror_2025   
snapshot                                                                      
2025-01-04 00:00:00          488.0         186.0       0.113264     0.07413   
2025-01-04 01:00:00          428.0         185.0       0.083006     0.07393   
2025-01-04 02:00:00          407.0         185.0       0.062000     0.07374   
2025-01-04 03:00:00          434.0         184.0       0.041521     0.07355   
2025-01-04 04:00:00          420.0         184.0       0.030790     0.07337   

                                                                               \
                    AL_solar_2025 AT_demand_2025 AT_hydro_2025 AT_onwind_2025   
snapshot                                                                        
2025-01-04 00:00:00           0.0         4405.0         219.0       0.201662   
2025-01-04 01:00:00           0.0         4312.0         219.0       0.192658   
2025-01-04 02:00:00           0.0         4211.0         219.0       0.187942   
2025-01-04 03:00:00           0.0         4299.0         219.0       0.170759   
2025-01-04 04:00:00           0.0         4619.0         219.0       0.158332   

                                               ...     residuals  \
                    AT_ror_2025 AT_solar_2025  ... UA_solar_2025   
snapshot                                       ...                 
2025-01-04 00:00:00    0.285512           0.0  ...           0.0   
2025-01-04 01:00:00    0.272398           0.0  ...           0.0   
2025-01-04 02:00:00    0.269771           0.0  ...           0.0   
2025-01-04 03:00:00    0.273883           0.0  ...           0.0   
2025-01-04 04:00:00    0.276763           0.0  ...           0.0   

                                                                               \
                    UK_demand_2025 UK_hydro_2025 UK_onwind_2025   UK_ror_2025   
snapshot                                                                        
2025-01-04 00:00:00   0.000000e+00  7.105427e-15            0.0  0.000000e+00   
2025-01-04 01:00:00   0.000000e+00  7.105427e-15            0.0  0.000000e+00   
2025-01-04 02:00:00   0.000000e+00  7.105427e-15            0.0  2.775558e-17   
2025-01-04 03:00:00   0.000000e+00  7.105427e-15            0.0  0.000000e+00   
2025-01-04 04:00:00   3.637979e-12  7.105427e-15            0.0  0.000000e+00   

                                                                            \
                    UK_solar_2025 XK_hydro_2025 XK_onwind_2025 XK_ror_2025   
snapshot                                                                     
2025-01-04 00:00:00           0.0  4.440892e-16   0.000000e+00         0.0   
2025-01-04 01:00:00           0.0  4.440892e-16   0.000000e+00         0.0   
2025-01-04 02:00:00           0.0  4.440892e-16  -3.469447e-18         0.0   
2025-01-04 03:00:00           0.0  4.440892e-16   1.734723e-18         0.0   
2025-01-04 04:00:00           0.0  4.440892e-16   0.000000e+00         0.0   

                                   
                    XK_solar_2025  
snapshot                           
2025-01-04 00:00:00           0.0  
2025-01-04 01:00:00           0.0  
2025-01-04 02:00:00           0.0  
2025-01-04 03:00:00           0.0  
2025-01-04 04:00:00           0.0  

[5 rows x 340 columns]

#### Extremes Comparison
Compare minimum and maximum values for the same example group. Comparing one reconstructed group to the full year would not be a like-for-like comparison.

In [304]:
def build_extreme_value_comparison(
    original: pd.DataFrame,
    reconstructed: pd.DataFrame,
) -> pd.DataFrame:
    """Compare min/max values between matching original and reconstructed tables."""
    comparison = pd.DataFrame(
        {
            "original_min": original.min(axis=0),
            "reconstructed_min": reconstructed.min(axis=0),
            "original_max": original.max(axis=0),
            "reconstructed_max": reconstructed.max(axis=0),
        }
    )

    comparison["min_error"] = (
        comparison["reconstructed_min"] - comparison["original_min"]
    )
    comparison["max_error"] = (
        comparison["reconstructed_max"] - comparison["original_max"]
    )
    comparison["max_error_%"] = (
        comparison["max_error"].div(comparison["original_max"]) * 100
    )

    return comparison


example_reconstructed_features = example_result.reconstructed.loc[
    :, FEATURE_COLUMNS
]
example_original_features = example_features.loc[
    :, example_reconstructed_features.columns
]

extreme_value_comparison = build_extreme_value_comparison(
    example_original_features,
    example_reconstructed_features,
)
extreme_value_comparison

,original_min,reconstructed_min,original_max,reconstructed_max,min_error,max_error,max_error_%
AL_demand_2025,372.0,374.0,658.0,639.0,2.000000e+00,-1.900000e+01,-2.887538e+00
AT_demand_2025,4117.0,4211.0,6899.0,6899.0,9.400000e+01,0.000000e+00,0.000000e+00
BA_demand_2025,553.0,586.0,1002.0,1002.0,3.300000e+01,0.000000e+00,0.000000e+00
BE_demand_2025,6886.0,7373.0,10172.0,10137.0,4.870000e+02,-3.500000e+01,-3.440818e-01
BG_demand_2025,2472.0,2472.0,3903.0,3745.0,0.000000e+00,-1.580000e+02,-4.048168e+00
...,...,...,...,...,...,...,...
UA_hydro_2025,554.0,560.0,619.0,603.0,6.000000e+00,-1.600000e+01,-2.584814e+00
UK_hydro_2025,38.0,38.0,43.0,43.0,0.000000e+00,-7.105427e-15,-1.652425e-14
XK_hydro_2025,3.0,3.0,14.0,14.0,-4.440892e-16,0.000000e+00,0.000000e+00
SE_hydro_2025,4250.0,4261.0,4647.0,4644.0,1.100000e+01,-3.000000e+00,-6.455778e-02


# Diagnostic Visualizations
[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/plot/#tsamplot)

This section presents assignment, weighting, and reconstruction-quality diagnostics.

In [305]:
# Setting visual theme here for persistence
plotly_theme: Final[str] = "ggplot2"

## Calendar Heatmap Of Original Days By Assigned Representative
Each day is colored by its assigned representative day.

The following cells prepare the calendar data and chart configuration. The helper functions are separated to make the plotting pipeline auditable and reusable for alternative clustering configurations.

### Calendar Data Preparation Helpers

In [306]:
def split_assignments_by_month(
    calendar_assignments: pd.DataFrame,
) -> list[pd.DataFrame]:
    """Split the day-level assignment table into one DataFrame per month."""
    return [
        calendar_assignments[calendar_assignments.index.month == month]
        for month in range(1, 13)
    ]


def add_calendar_coordinates(month_assignments: pd.DataFrame) -> pd.DataFrame:
    """Add day, weekday, and calendar row coordinates for one month."""
    month_assignments = month_assignments.copy()
    month_assignments["day"] = month_assignments.index.day
    month_assignments["weekday"] = month_assignments.index.day_name()
    month_assignments["weekday_num"] = month_assignments.index.weekday

    first_weekday = month_assignments.index.min().weekday()
    month_assignments["week_row"] = (
        month_assignments["day"] + first_weekday - 1
    ) // 7
    return month_assignments


def build_representative_color_grid(
    month_assignments: pd.DataFrame,
) -> pd.DataFrame:
    """Pivot one month into a week-row x weekday color-id grid."""
    return month_assignments.pivot(
        index="week_row",
        columns="weekday_num",
        values="representative_color_id",
    )


def prep_month_for_plotting(
    month_assignments: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return month-level plotting data and its color-id grid."""
    prepared_month = add_calendar_coordinates(month_assignments)
    return prepared_month, build_representative_color_grid(prepared_month)


Prepare one day-level assignment table and split it into month-level plotting inputs.

In [307]:
calendar_assignments = day_assignments_df.loc[:, ["representative_id"]].copy()
calendar_assignments.index = calendar_assignments.index.normalize()

# Plotly heatmaps need numeric color values; keep the real ID for hover labels.
representative_ids = calendar_assignments["representative_id"].drop_duplicates()
representative_color_map = {
    representative_id: color_id
    for color_id, representative_id in enumerate(representative_ids)
}
calendar_assignments["representative_color_id"] = calendar_assignments[
    "representative_id"
].map(representative_color_map)

monthly_calendar_assignments = split_assignments_by_month(calendar_assignments)
processed_monthly_assignments: list[tuple[pd.DataFrame, pd.DataFrame]] = [
    prep_month_for_plotting(month) for month in monthly_calendar_assignments
]

### Calendar Plotting Constants

In [308]:
CALENDAR_ROWS: Final[int] = 4
CALENDAR_COLS: Final[int] = 3
WEEKDAY_NUMBERS: Final[list[int]] = list(range(7))
WEEKEND_WEEKDAY_NUMBERS: Final[set[int]] = {5, 6}
WEEKDAY_LABELS: Final[list[str]] = ["Mo", "Tu", "We", "Th", "Fr", "Sa", "Su"]
MONTH_TITLES: Final[list[str]] = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
REPRESENTATIVE_PALETTE: Final[list[str]] = (
    px.colors.qualitative.Plotly
    + px.colors.qualitative.Set3
    + px.colors.qualitative.Dark24
)


### Calendar Color Scale Helpers

In [309]:
def get_representative_color_ids(
    calendar_assignments: pd.DataFrame,
) -> list[int]:
    """Return representative color IDs in display order."""
    return [
        int(color_id)
        for color_id in sorted(
            calendar_assignments["representative_color_id"].unique()
        )
    ]


def build_discrete_representative_colorscale(
    representative_color_ids: list[int],
) -> tuple[list[tuple[float, str]], float, float]:
    """Build a Plotly colorscale where each representative has one color."""
    zmin = min(representative_color_ids) - 0.5
    zmax = max(representative_color_ids) + 0.5
    representative_colors = {
        color_id: REPRESENTATIVE_PALETTE[i % len(REPRESENTATIVE_PALETTE)]
        for i, color_id in enumerate(representative_color_ids)
    }

    colorscale: list[tuple[float, str]] = []
    for color_id in representative_color_ids:
        color = representative_colors[color_id]
        colorscale.append(((color_id - 0.5 - zmin) / (zmax - zmin), color))
        colorscale.append(((color_id + 0.5 - zmin) / (zmax - zmin), color))

    return colorscale, zmin, zmax


### Single-Month Heatmap Helpers

In [310]:
def build_day_text_grid(month_data: pd.DataFrame) -> pd.DataFrame:
    """Return day numbers formatted for display inside calendar cells."""
    day_number_grid = month_data.pivot(
        index="week_row",
        columns="weekday_num",
        values="day",
    ).reindex(columns=WEEKDAY_NUMBERS)
    return day_number_grid.map(
        lambda value: "" if pd.isna(value) else str(int(value))
    )


def build_calendar_hover_text_grid(month_data: pd.DataFrame) -> pd.DataFrame:
    """Return hover labels with weekday and representative ID."""
    representative_grid = month_data.pivot(
        index="week_row",
        columns="weekday_num",
        values="representative_id",
    ).reindex(columns=WEEKDAY_NUMBERS)

    hover_text_grid = representative_grid.copy()
    for weekday_num, weekday_label in enumerate(WEEKDAY_LABELS):
        hover_text_grid[weekday_num] = representative_grid[weekday_num].map(
            lambda representative_id: (
                ""
                if pd.isna(representative_id)
                else f"{weekday_label}<br>Representative: {representative_id}"
            )
        )
    return hover_text_grid


def plot_calendar_month(
    month_data: pd.DataFrame,
    month_grid: pd.DataFrame,
    representative_color_ids: list[int],
) -> go.Heatmap:
    """Build one month-calendar heatmap trace colored by representative day."""
    calendar_grid = month_grid.reindex(columns=WEEKDAY_NUMBERS)
    colorscale, zmin, zmax = build_discrete_representative_colorscale(
        representative_color_ids
    )

    return go.Heatmap(
        z=calendar_grid.to_numpy(),
        x=WEEKDAY_NUMBERS,
        y=calendar_grid.index,
        text=build_day_text_grid(month_data).to_numpy(),
        customdata=build_calendar_hover_text_grid(month_data).to_numpy(),
        texttemplate="%{text}",
        textfont={"size": 13, "color": "#333"},
        colorscale=colorscale,
        zmin=zmin,
        zmax=zmax,
        hovertemplate="Day %{text}<br>%{customdata}<extra></extra>",
        xgap=2,
        ygap=2,
        showscale=False,
    )


Build one heatmap trace per month.

In [311]:
representative_color_ids = get_representative_color_ids(calendar_assignments)
calendar_heatmaps: list[go.Heatmap] = [
    plot_calendar_month(month_data, month_grid, representative_color_ids)
    for month_data, month_grid in processed_monthly_assignments
]


### Full-Year Calendar Assembly Helpers

In [312]:
def get_calendar_subplot_positions(
    rows: int = CALENDAR_ROWS, cols: int = CALENDAR_COLS
) -> list[tuple[int, int]]:
    """Return Plotly subplot positions in row-major order."""
    return [
        (row, col) for row in range(1, rows + 1) for col in range(1, cols + 1)
    ]


def create_calendar_subplot_figure(
    heatmaps: list[go.Heatmap],
) -> tuple[go.Figure, list[tuple[int, int]]]:
    """Create the 12-month subplot figure and add monthly heatmap traces."""
    fig = make_subplots(
        rows=CALENDAR_ROWS,
        cols=CALENDAR_COLS,
        subplot_titles=MONTH_TITLES,
    )
    subplot_positions = get_calendar_subplot_positions()
    fig.add_traces(
        data=heatmaps,
        rows=[row for row, _ in subplot_positions],
        cols=[col for _, col in subplot_positions],
    )
    return fig, subplot_positions


def add_weekend_borders(
    fig: go.Figure,
    processed_months: list[tuple[pd.DataFrame, pd.DataFrame]],
    subplot_positions: list[tuple[int, int]],
) -> None:
    """Mark weekend cells while preserving their representative fill color."""
    for (month_data, _), (row, col) in zip(processed_months, subplot_positions):
        weekend_days = month_data[
            month_data["weekday_num"].isin(WEEKEND_WEEKDAY_NUMBERS)
        ]
        for _, day in weekend_days.iterrows():
            fig.add_shape(
                type="rect",
                x0=day["weekday_num"] - 0.5,
                x1=day["weekday_num"] + 0.5,
                y0=day["week_row"] - 0.5,
                y1=day["week_row"] + 0.5,
                line=dict(color="rgba(35, 35, 35, 0.75)", width=2),
                fillcolor="rgba(0, 0, 0, 0)",
                layer="above",
                row=row,
                col=col,
            )


### Full-Year Calendar Styling Helpers

In [313]:
def style_calendar_figure(fig: go.Figure, template: str) -> None:
    """Apply shared layout and axis styling to the full-year calendar."""
    fig.update_layout(
        title={
            "text": "2025 representative-day assignment calendar",
            "x": 0.5,
            "font_size": 20,
            "y": 0.99,
        },
        template=template,
        width=None,
        height=900,
        plot_bgcolor="white",
        margin={"l": 20, "r": 20, "t": 120, "b": 20},
        autosize=True,
    )
    fig.update_annotations(yshift=25)
    fig.update_xaxes(
        side="top",
        tickmode="array",
        tickvals=WEEKDAY_NUMBERS,
        ticktext=WEEKDAY_LABELS,
        showticklabels=True,
        ticks="",
        showline=False,
        showgrid=False,
        zeroline=False,
    )
    fig.update_yaxes(
        ticks="",
        showline=False,
        autorange="reversed",
        showgrid=False,
        zeroline=False,
        showticklabels=False,
    )


def build_representative_calendar_figure(
    processed_months: list[tuple[pd.DataFrame, pd.DataFrame]],
    heatmaps: list[go.Heatmap],
    template: str,
) -> go.Figure:
    """Build the full-year representative assignment calendar."""
    fig, subplot_positions = create_calendar_subplot_figure(heatmaps)
    add_weekend_borders(fig, processed_months, subplot_positions)
    style_calendar_figure(fig, template)
    return fig


Build and display the calendar visualization.

In [314]:
calendar_fig = build_representative_calendar_figure(
    processed_monthly_assignments, calendar_heatmaps, plotly_theme
)
calendar_fig.show()

## Representative Weight Heatmap
This heatmap shows how much of each month/day-type group is represented by each selected representative day.

The denominator is group-specific: a working representative is divided by the number of working days in that month, while a non-working representative is divided by the number of non-working days in that month.

**Display setup.** Define the month order and representative row-sorting rules. Heatmap rows are derived from the representatives that are actually present in `representative_days`.

In [315]:
REPRESENTATIVE_DAY_TYPE_SORT_ORDER: Final[dict[str, int]] = {
    "working": 0,
    "non-working": 1,
}


def sort_representative_group(representative_group: str) -> tuple[int, int]:
    """Sort representative rows by day type and numeric cluster ID."""
    day_type, cluster_label = representative_group.rsplit("_c", maxsplit=1)
    return (
        REPRESENTATIVE_DAY_TYPE_SORT_ORDER.get(day_type, 99),
        int(cluster_label),
    )


month_order: Final[list[str]] = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
month_name_by_number = dict(zip(range(1, 13), month_order))

**Build the chart table.** Start from `representative_days`, because this chart has one value per selected representative day.

In [316]:
representative_weight_summary = representative_days.copy()
representative_weight_summary["month_name"] = representative_weight_summary[
    "month"
].map(month_name_by_number)
representative_weight_summary["representative_group"] = (
    representative_weight_summary["day_type"]
    + "_c"
    + representative_weight_summary["cluster_id"].astype(str)
)

# Denominator: days in the same month and day type.
representative_weight_summary["group_days"] = (
    representative_weight_summary.groupby(["month", "day_type"])[
        "cluster_weight"
    ].transform("sum")
)
representative_weight_summary["group_share_pct"] = (
    representative_weight_summary["cluster_weight"]
    .div(representative_weight_summary["group_days"])
    .mul(100)
)

assert (
    representative_weight_summary.groupby(["month", "day_type"])[
        "group_share_pct"
    ]
    .sum()
    .round(6)
    .eq(100)
    .all()
)

representative_weight_summary.head()

,selected_medoid_date,representative_id,group_id,month,day_type,cluster_id,cluster_weight,month_name,representative_group,group_days,group_share_pct
0,2025-01-04,2025_1_non-working_c0,2025_1_non-working,1,non-working,0,6,January,non-working_c0,8,75.000000
1,2025-01-07,2025_1_working_c0,2025_1_working,1,working,0,8,January,working_c0,23,34.782609
2,2025-01-16,2025_1_working_c2,2025_1_working,1,working,2,5,January,working_c2,23,21.739130
3,2025-01-22,2025_1_working_c1,2025_1_working,1,working,1,5,January,working_c1,23,21.739130
4,2025-01-25,2025_1_non-working_c1,2025_1_non-working,1,non-working,1,2,January,non-working_c1,8,25.000000


**Prepare heatmap matrices.** Pivot the representative-day summary into representative-group by month matrices. The row count follows the clustering configuration used to create `representative_days`.

In [317]:
representative_group_order = sorted(
    representative_weight_summary["representative_group"].unique(),
    key=sort_representative_group,
)

representative_group_share_pct = representative_weight_summary.pivot(
    index="representative_group",
    columns="month_name",
    values="group_share_pct",
).reindex(index=representative_group_order, columns=month_order)

representative_group_assigned_days = representative_weight_summary.pivot(
    index="representative_group",
    columns="month_name",
    values="cluster_weight",
).reindex(index=representative_group_order, columns=month_order)

representative_group_total_days = representative_weight_summary.pivot(
    index="representative_group",
    columns="month_name",
    values="group_days",
).reindex(index=representative_group_order, columns=month_order)

representative_weight_text = representative_group_share_pct.map(
    lambda value: "" if pd.isna(value) else f"{value:.1f}%"
)

# Hover keeps both the percentage and the raw day counts.
representative_weight_hover_data = [
    [
        [assigned_days, total_days]
        for assigned_days, total_days in zip(assigned_row, total_row)
    ]
    for assigned_row, total_row in zip(
        representative_group_assigned_days.to_numpy(),
        representative_group_total_days.to_numpy(),
    )
]

**Configure and render the heatmap.** Define dynamic display settings and render the prepared matrix as a Plotly heatmap.

In [318]:
# Row count changes when the cluster-count policy changes.
representative_group_count = len(representative_group_share_pct)

# Hide in-cell labels when the matrix becomes too dense to read.
show_representative_weight_labels = representative_group_share_pct.size <= 240

# Give additional vertical space for additional representative groups.
representative_weight_heatmap_height = max(
    500, 120 + 42 * representative_group_count
)

# Slightly reduce y-axis labels once there are many representative groups.
representative_weight_y_tick_size = (
    14 if representative_group_count <= 10 else 11
)

fig = px.imshow(
    representative_group_share_pct,
    aspect="auto",
    labels={
        "x": "Month",
        "y": "Representative group",
        "color": "% of month/day-type group",
    },
)

fig.update_traces(
    customdata=representative_weight_hover_data,
    text=representative_weight_text.to_numpy(),
    texttemplate="%{text}" if show_representative_weight_labels else "",
    hovertemplate=(
        "Month: %{x}<br>"
        "Representative group: %{y}<br>"
        "Share of month/day-type group: %{z:.1f}%<br>"
        "Days assigned: %{customdata[0]:.0f}<br>"
        "Days in month/day-type group: %{customdata[1]:.0f}"
        "<extra></extra>"
    ),
)
fig.update_layout(
    title={
        "text": "Share of each month/day-type group assigned to each representative",
        "x": 0.5,
        "font_size": 20,
    },
    autosize=True,
    width=None,
    height=representative_weight_heatmap_height,
    margin={"l": 20, "r": 20, "t": 60, "b": 40},
    coloraxis_showscale=False,
)
fig.update_xaxes(title_font_size=18, tickfont={"size": 14})
fig.update_yaxes(
    title_font_size=18, tickfont={"size": representative_weight_y_tick_size}
)

fig.show()

## Cluster Accuracy Overview
This section compresses the 24 group-level TSAM results into one quality summary.

Each row is one month/day-type group. `weighted_rmse` is the primary ranking metric, and `weighted_rmse_duration` is retained as a secondary duration-curve diagnostic.

**Build the summary table.** Collect one weighted accuracy score per TSAM group, then join the month/day-type metadata used for plotting and hover labels.

In [319]:
group_accuracy_metrics = pd.DataFrame.from_dict(
    {
        group_id: {
            "weighted_rmse": result.accuracy.weighted_rmse,
            "weighted_rmse_duration": result.accuracy.weighted_rmse_duration,
        }
        for group_id, result in tsam_results_by_group.items()
    },
    orient="index",
).rename_axis("group_id")

group_metadata = representative_days.groupby("group_id").agg(
    month=("month", "first"),
    day_type=("day_type", "first"),
    n_days=("cluster_weight", "sum"),
    n_clusters=("cluster_id", "nunique"),
)

group_accuracy = group_accuracy_metrics.join(group_metadata).sort_values(
    "weighted_rmse", ascending=False
)

assert len(group_accuracy) == len(tsam_results_by_group)
assert (
    group_accuracy[["month", "day_type", "n_days", "n_clusters"]]
    .notna()
    .all()
    .all()
)

group_accuracy

,weighted_rmse,weighted_rmse_duration,month,day_type,n_days,n_clusters
group_id,,,,,,
2025_3_non-working,0.251804,0.183754,3,non-working,10,2
2025_11_non-working,0.249132,0.194370,11,non-working,10,2
2025_9_non-working,0.247330,0.160952,9,non-working,8,2
2025_4_non-working,0.247268,0.169703,4,non-working,8,2
2025_1_non-working,0.243749,0.172376,1,non-working,8,2
2025_2_non-working,0.241644,0.164761,2,non-working,8,2
2025_10_non-working,0.239998,0.184869,10,non-working,8,2
2025_8_non-working,0.229720,0.170125,8,non-working,10,2
2025_5_non-working,0.221006,0.153326,5,non-working,9,2


**Plot the ranked overview.** Sort the groups from best to worst so the largest bars identify the groups that warrant further inspection.

In [320]:
group_accuracy_plot = group_accuracy.reset_index().sort_values(
    "weighted_rmse",
    ascending=True,
)

fig = px.bar(
    group_accuracy_plot,
    x="weighted_rmse",
    y="group_id",
    color="day_type",
    orientation="h",
    custom_data=[
        "month",
        "day_type",
        "n_days",
        "n_clusters",
        "weighted_rmse_duration",
    ],
    labels={
        "weighted_rmse": "Weighted RMSE",
        "group_id": "Group",
        "day_type": "Day type",
    },
)

fig.update_traces(
    hovertemplate=(
        "Group: %{y}<br>"
        "Month: %{customdata[0]}<br>"
        "Day type: %{customdata[1]}<br>"
        "Original days: %{customdata[2]:.0f}<br>"
        "Clusters: %{customdata[3]:.0f}<br>"
        "Weighted RMSE: %{x:.4f}<br>"
        "Weighted duration RMSE: %{customdata[4]:.4f}"
        "<extra></extra>"
    ),
)
fig.update_layout(
    title={
        "text": "Weighted RMSE by group",
        "x": 0.5,
        "font_size": 20,
    },
    autosize=True,
    width=None,
    height=700,
    margin={"l": 20, "r": 20, "t": 40, "b": 20},
)
fig.update_xaxes(title_font_size=18, tickfont={"size": 14})
fig.update_yaxes(title_font_size=18, tickfont={"size": 12})

fig.show()

## Group-Level TSAM Diagnostic Drilldowns
These widgets support inspection of one TSAM group at a time without duplicating each chart 24 times.

The group dropdown selects the month/day-type aggregation result. For plots with feature dropdowns, capacity factors, demand, and hydro are separated because they use different scales.

Make sure all groups have the same number of columns/features:

In [321]:
reference_group = group_ids[0]
reference_columns = tsam_results_by_group[
    reference_group
].cluster_representatives.columns

for group in group_ids:
    columns = tsam_results_by_group[group].cluster_representatives.columns
    if not columns.equals(reference_columns):
        raise ValueError(f"{group}: columns differ from {reference_group}")

Since we are now working with ALL countries, and thus the number of features increases to N_FEATURES_PER_COUNTRY * N_COUNTRIES, we need a granularity controller, thus we are adding an additional dropdown to the TSAM drilldown charts – country.

In [322]:
tsam_results_by_group['2025_1_working'].cluster_representatives

AL_demand_2025  AL_hydro_2025  AL_onwind_2025  AL_ror_2025  \
  timestep                                                               
0 0                  451.0          170.0        0.100100      0.06781   
  1                  414.0          170.0        0.114121      0.06776   
  2                  395.0          169.0        0.122708      0.06771   
  3                  389.0          169.0        0.127203      0.06766   
  4                  405.0          169.0        0.131076      0.06761   
...                    ...            ...             ...          ...   
4 19                 580.0         1377.0        0.000000      0.55019   
  20                 564.0         1363.0        0.000000      0.54456   
  21                 534.0         1349.0        0.000000      0.53893   
  22                 511.0         1334.0        0.000000      0.53299   
  23                 468.0         1319.0        0.000000      0.52725   

            AL_solar_2025  AT_demand_2025  AT_hydro_2025  AT_onwind_2025  \
  timestep                                                                 
0 0                   0.0          4029.0          218.0        0.120127   
  1                   0.0          3944.0          218.0        0.128518   
  2                   0.0          3886.0          218.0        0.127268   
  3                   0.0          4018.0          218.0        0.102271   
  4                   0.0          4561.0          218.0        0.080899   
...                   ...             ...            ...             ...   
4 19                  0.0          5915.0          196.0        0.035923   
  20                  0.0          5512.0          196.0        0.027180   
  21                  0.0          5428.0          196.0        0.019905   
  22                  0.0          5080.0          196.0        0.015688   
  23                  0.0          4826.0          196.0        0.017822   

            AT_ror_2025  AT_solar_2025  ...  UA_solar_2025  UK_demand_2025  \
  timestep                              ...                                  
0 0            0.383636            0.0  ...            0.0         19638.0   
  1            0.308894            0.0  ...            0.0         19112.0   
  2            0.389414            0.0  ...            0.0         18445.0   
  3            0.391878            0.0  ...            0.0         18071.0   
  4            0.387548            0.0  ...            0.0         17039.0   
...                 ...            ...  ...            ...             ...   
4 19           0.529290            0.0  ...            0.0         33002.0   
  20           0.508151            0.0  ...            0.0         31785.0   
  21           0.490472            0.0  ...            0.0         29461.0   
  22           0.476434            0.0  ...            0.0         26292.0   
  23           0.469822            0.0  ...            0.0         23814.0   

            UK_hydro_2025  UK_onwind_2025  UK_ror_2025  UK_solar_2025  \
  timestep                                                              
0 0                  42.0        0.598620      0.19147            0.0   
  1                  42.0        0.598569      0.19146            0.0   
  2                  42.0        0.587477      0.19143            0.0   
  3                  42.0        0.580186      0.19140            0.0   
  4                  42.0        0.563502      0.19139            0.0   
...                   ...             ...          ...            ...   
4 19                 38.0        0.223568      0.16998            0.0   
  20                 38.0        0.268949      0.16994            0.0   
  21                 38.0        0.312441      0.16990            0.0   
  22                 38.0        0.354778      0.16986            0.0   
  23                 38.0        0.356740      0.16981            0.0   

            XK_hydro_2025  XK_onwind_2025  XK_ror_2025  XK_solar_2025  
  timestep             

Since the features are on different scales, group features into 3 categories: CF, demand and hydro:

In [323]:
def feature_group(col: str) -> str:
    tokens = col.split("_")
    if {"solar", "onwind", "ror"} & set(tokens):
        return "capacity_factors"
    if "demand" in tokens:
        return "demand"
    if "hydro" in tokens:
        return "hydro"
    raise ValueError(f"Encountered an unknown column: {col}")


FEATURE_GROUPS: Final[dict[str, list[str]]] = {
    "capacity_factors": [],
    "demand": [],
    "hydro": [],
}
for col in reference_columns:
    FEATURE_GROUPS[feature_group(col)].append(col)

capacity_factor_cols = FEATURE_GROUPS["capacity_factors"]
for group in group_ids:
    reps = tsam_results_by_group[group].cluster_representatives
    values = reps.loc[:, capacity_factor_cols]
    is_in_range = values.ge(0.0) & values.le(1.0)
    invalid_cols = is_in_range.columns[~is_in_range.all()].tolist()
    if invalid_cols:
        raise ValueError(f"{group}: capacity factors out of range in {invalid_cols}")

In [324]:
FEATURE_GROUPS

{'capacity_factors': ['AL_onwind_2025',
  'AL_ror_2025',
  'AL_solar_2025',
  'AT_onwind_2025',
  'AT_ror_2025',
  'AT_solar_2025',
  'BA_onwind_2025',
  'BA_ror_2025',
  'BA_solar_2025',
  'BE_onwind_2025',
  'BE_ror_2025',
  'BE_solar_2025',
  'BG_onwind_2025',
  'BG_ror_2025',
  'BG_solar_2025',
  'CH_onwind_2025',
  'CH_ror_2025',
  'CH_solar_2025',
  'CZ_onwind_2025',
  'CZ_ror_2025',
  'CZ_solar_2025',
  'DE_onwind_2025',
  'DE_ror_2025',
  'DE_solar_2025',
  'DK_onwind_2025',
  'DK_solar_2025',
  'EE_onwind_2025',
  'EE_ror_2025',
  'EE_solar_2025',
  'ES_onwind_2025',
  'ES_ror_2025',
  'ES_solar_2025',
  'FI_onwind_2025',
  'FI_ror_2025',
  'FI_solar_2025',
  'FR_onwind_2025',
  'FR_ror_2025',
  'FR_solar_2025',
  'GR_onwind_2025',
  'GR_ror_2025',
  'GR_solar_2025',
  'HR_onwind_2025',
  'HR_ror_2025',
  'HR_solar_2025',
  'HU_onwind_2025',
  'HU_ror_2025',
  'HU_solar_2025',
  'IE_onwind_2025',
  'IE_ror_2025',
  'IE_solar_2025',
  'IT_onwind_2025',
  'IT_ror_2025',
  'IT_so

## Plotting setup
These helper functions keep widget ordering, figure size, and Plotly styling consistent across all TSAM drilldown charts.

Each chart receives independent dropdown widgets. Reusing a single widget object across multiple charts would cause all connected outputs to update together.

In [325]:
# Reuse the canonical chronological order defined in the data-collection section.
GROUP_OPTIONS: Final[list[str]] = list(group_ids)

def match_features_to_country(features, country):
    """Index features by the chosen country in the dropdown"""
    return [f for f in features if f.startswith(country)]


def make_group_dropdown() -> widgets.Dropdown:
    """Create a fresh chronological group selector for one chart."""
    return widgets.Dropdown(
        options=GROUP_OPTIONS,
        description="Group:",
        layout=widgets.Layout(width="360px"),
    )


def make_feature_dropdown() -> widgets.Dropdown:
    """Create a fresh feature-group selector for one chart."""
    return widgets.Dropdown(
        options=FEATURE_GROUPS,
        description="Features:",
        layout=widgets.Layout(width="320px"),
    )

def make_country_dropdown() -> widgets.Dropdown:
    """Create a fresh country-group selector for one chart."""
    return widgets.Dropdown(
        options=all_countries,
        description="Countries:",
        layout=widgets.Layout(width="320px"),
    )


def style_interactive_tsam_figure(fig: go.Figure, title: str) -> go.Figure:
    """Apply notebook-friendly sizing to interactive TSAM figures."""
    fig.update_layout(
        template=plotly_theme,
        title={
            "text": title,
            "x": 0.5,
            "xanchor": "center",
            "yanchor": "top",
            "font_size": 20,
        },
        autosize=True,
        width=None,
        height=720,
        margin={"l": 40, "r": 30, "t": 120, "b": 70},
        legend={
            "yanchor": "top",
            "y": 1,
            "xanchor": "left",
            "x": 1.02,
        },
    )
    # TSAM uses Plotly facet annotations for multi-column plots.
    fig.update_annotations(font_size=13)
    return fig


def display_group_plot(plot_function) -> None:
    """Display one group-only TSAM plot with its own group dropdown."""
    group_dropdown = make_group_dropdown()
    output = widgets.interactive_output(
        plot_function,
        {"group_id": group_dropdown},
    )
    # Keep controls and output as one notebook output block.
    display(widgets.VBox([widgets.HBox([group_dropdown]), output]))


def display_group_feature_plot(plot_function) -> None:
    """Display one group-and-feature TSAM plot with independent dropdowns."""
    # Fresh widgets prevent one chart's controls from updating other charts.
    group_dropdown = make_group_dropdown()
    feature_dropdown = make_feature_dropdown()
    countries_dropdown = make_country_dropdown()
    output = widgets.interactive_output(
        plot_function,
        {
            "group_id": group_dropdown,
            "feature_columns": feature_dropdown,
            "country": countries_dropdown
        },
    )
    display(
        widgets.VBox([widgets.HBox([group_dropdown, feature_dropdown, countries_dropdown]), output])
    )

## Cluster Representatives
Representative daily profiles selected by TSAM. This plot shows the typical days that replace original days in each group.

In [326]:
def show_cluster_representatives(
    group_id: str, feature_columns: list[str], country: str
) -> None:
    """Plot representative daily profiles for the selected group and feature set."""
    result = tsam_results_by_group[group_id]
    feature_columns = match_features_to_country(features=feature_columns, country=country)
    fig = result.plot.cluster_representatives(columns=feature_columns, title="")
    fig = style_interactive_tsam_figure(
        fig,
        title=f"Cluster representative profiles: {group_id}",
    )
    fig.show(config={"responsive": True})


display_group_feature_plot(show_cluster_representatives)

## Cluster Members
Original member days are grouped by cluster, with the selected representative highlighted. The TSAM figure retains its internal cluster slider.

In [329]:
def show_cluster_members(group_id: str, feature_columns: list[str], country: str) -> None:
    """Plot original member days and highlighted representatives by cluster."""
    result = tsam_results_by_group[group_id]
    feature_columns = match_features_to_country(features=feature_columns, country=country)
    fig = result.plot.cluster_members(
        columns=feature_columns,
        slider="cluster",
        title="",
    )
    # Cluster selection stays inside the TSAM/Plotly figure slider.
    fig = style_interactive_tsam_figure(
        fig,
        title=f"Cluster members: {group_id}",
    )
    fig.show(config={"responsive": True})


display_group_feature_plot(show_cluster_members)

## Cluster Weights
This chart shows how many original days each representative day replaces inside the selected group.

In [255]:
def show_cluster_weights(group_id: str) -> None:
    """Plot how many original days each representative day replaces."""
    result = tsam_results_by_group[group_id]
    fig = result.plot.cluster_weights(title="")
    fig = style_interactive_tsam_figure(
        fig,
        title=f"Cluster weights: {group_id}",
    )
    fig.show(config={"responsive": True})


display_group_plot(show_cluster_weights)

## Cluster Accuracy
This chart shows per-feature reconstruction error for the selected group.

In [256]:
def show_cluster_accuracy(group_id: str) -> None:
    """Plot TSAM reconstruction accuracy metrics for the selected group."""
    result = tsam_results_by_group[group_id]
    fig = result.plot.accuracy(title="")
    fig = style_interactive_tsam_figure(
        fig,
        title=f"Cluster accuracy: {group_id}",
    )
    fig.show(config={"responsive": True})


display_group_plot(show_cluster_accuracy)

## Full-Resolution Comparison
This chart compares original and reconstructed time series for the selected group and feature set.

In [ ]:
def show_full_resolution_comparison(
    group_id: str, feature_columns: list[str], country: str
) -> None:
    """Plot original versus reconstructed time series for selected features."""
    result = tsam_results_by_group[group_id]
    feature_columns = match_features_to_country(features=feature_columns, country=country)
    fig = result.plot.compare(columns=feature_columns, title="")
    fig = style_interactive_tsam_figure(
        fig,
        title=f"Original vs reconstructed: {group_id}",
    )
    fig.show(config={"responsive": True})


display_group_feature_plot(show_full_resolution_comparison)

## Residuals
This chart shows pointwise reconstruction error for the selected group and feature set.

In [ ]:
def show_cluster_residuals(group_id: str, feature_columns: list[str], country: str) -> None:
    """Plot pointwise reconstruction residuals for selected features."""
    result = tsam_results_by_group[group_id]
    feature_columns = match_features_to_country(features=feature_columns, country=country)
    fig = result.plot.residuals(columns=feature_columns, title="")
    fig = style_interactive_tsam_figure(
        fig,
        title=f"Residuals: {group_id}",
    )
    fig.show(config={"responsive": True})


display_group_feature_plot(show_cluster_residuals)

# Saving Results
Save the three portable tables that define the reduced dataset:

- `reduced_hourly_df`: selected representative hours, prepared for downstream model input
- `representative_days`: one row per selected medoid day
- `day_assignments_df`: original day to representative-day mapping

The raw `tsam_results_by_group` dictionary remains in memory because it contains Python result objects rather than a stable flat table.

In [259]:
output_path: Path = cur_location / "outputs" / "approach_1"
output_path.mkdir(parents=True, exist_ok=True)

# Reset timestamp/date indexes so CSV readers get explicit columns.
tables_to_save: Final[dict[str, pd.DataFrame]] = {
    "reduced_hourly_df.csv": reduced_hourly_df.reset_index(),
    "representative_days.csv": representative_days,
    "day_assignments_df.csv": day_assignments_df.reset_index(),
}

for file_name, table in tables_to_save.items():
    table.to_csv(output_path / file_name, index=False)

saved_files = pd.DataFrame(
    [
        {
            "file_name": file_name,
            "rows": len(table),
            "columns": len(table.columns),
        }
        for file_name, table in tables_to_save.items()
    ]
)
saved_files

,file_name,rows,columns
0,reduced_hourly_df.csv,2016,181
1,representative_days.csv,84,7
2,day_assignments_df.csv,365,5
